In [ ]:
import warnings; warnings.filterwarnings('ignore')
import re, time, urllib.request, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from html.parser import HTMLParser
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_curve, f1_score
from sklearn.preprocessing import StandardScaler

pages = [
    (0, 'https://en.wikipedia.org/wiki/Stock_market',                 1, 'Stock market'),
    (1, 'https://en.wikipedia.org/wiki/Nasdaq',                       1, 'Nasdaq'),
    (2, 'https://en.wikipedia.org/wiki/Dow_Jones_Industrial_Average', 1, 'Dow Jones'),
    (3, 'https://en.wikipedia.org/wiki/S%26P_500',                    1, 'S&P 500'),
    (4, 'https://en.wikipedia.org/wiki/Investment',                   1, 'Investment'),
    (5, 'https://en.wikipedia.org/wiki/Association_football',         0, 'Football'),
    (6, 'https://en.wikipedia.org/wiki/Basketball',                   0, 'Basketball'),
    (7, 'https://en.wikipedia.org/wiki/Olympic_Games',                0, 'Olympics'),
    (8, 'https://en.wikipedia.org/wiki/Tennis',                       0, 'Tennis'),
    (9, 'https://en.wikipedia.org/wiki/Baseball',                     0, 'Baseball'),
]
page_urls   = [p[1] for p in pages]
page_labels = [p[2] for p in pages]
page_names  = [p[3] for p in pages]
page_paths  = [u.replace('https://en.wikipedia.org', '') for u in page_urls]
N = len(pages)

In [ ]:
class WikiParser(HTMLParser):
    def __init__(self):
        super().__init__()
        self.text, self.links, self._in_p = [], [], False
    def handle_starttag(self, tag, attrs):
        if tag == 'p': self._in_p = True
        elif tag == 'a':
            href = dict(attrs).get('href', '')
            if href.startswith('/wiki/') and ':' not in href: self.links.append(href)
    def handle_endtag(self, tag):
        if tag == 'p': self._in_p = False
    def handle_data(self, data):
        if self._in_p: self.text.append(data)

def fetch(url):
    try:
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        html = urllib.request.urlopen(req, timeout=15).read().decode('utf-8', errors='ignore')
        p = WikiParser(); p.feed(html)
        return ' '.join(p.text), p.links
    except Exception as e:
        print('  error:', e); return '', []

texts, links = [], []
for pid, url, label, name in pages:
    t, l = fetch(url)
    texts.append(t); links.append(l)
    print(f'  node {pid} {name}: {len(t)} chars, {len(l)} links')
    time.sleep(0.5)

A = np.zeros((N, N))
for i, llist in enumerate(links):
    ls = set(llist)
    for j, path in enumerate(page_paths):
        if i != j and path in ls: A[i, j] = 1.0
print('total edges:', int(A.sum()))

In [ ]:
def pagerank(A, d=0.85, iters=100):
    n = A.shape[0]
    M = A.T.copy()
    cs = M.sum(axis=0); cs[cs==0] = 1
    M /= cs
    pr = np.ones(n) / n
    for _ in range(iters):
        new = (1-d)/n + d * (M @ pr)
        if np.linalg.norm(new-pr, 1) < 1e-6: break
        pr = new
    return pr

def hits(A, iters=100):
    auth = np.ones(A.shape[0])
    for _ in range(iters):
        new_auth = A.T @ (A @ auth)
        norm = np.linalg.norm(new_auth)
        new_auth /= (norm + 1e-10)
        if np.linalg.norm(new_auth - auth) < 1e-6: break
        auth = new_auth
    return auth

pr_scores = pagerank(A)
hits_scores = hits(A)

for i, (name, label) in enumerate(zip(page_names, page_labels)):
    print(f'  {name:28s} rel={label}  PR={pr_scores[i]:.4f}  HITS={hits_scores[i]:.4f}')

In [ ]:
query = ['reuters', 'stocks', 'investment', 'market', 'prices', 'financial', 'trading', 'fund']
vec = TfidfVectorizer(vocabulary=query, stop_words='english')
tfidf_scores = np.array(vec.fit_transform(texts).sum(axis=1)).flatten()

X = np.column_stack([pr_scores, hits_scores, tfidf_scores])
y = np.array(page_labels)

df = pd.DataFrame({'page': page_names, 'relevant': y,
                   'pagerank': X[:,0].round(4), 'hits': X[:,1].round(4), 'tfidf': X[:,2].round(4)})
print(df.to_string(index=False))

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
sc = StandardScaler()
X_tr = sc.fit_transform(X_tr); X_te = sc.transform(X_te)

clf = LogisticRegression(max_iter=1000, random_state=7)
clf.fit(X_tr, y_tr)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

prec_tr, rec_tr, _ = precision_recall_curve(y_tr, clf.predict_proba(X_tr)[:,1])
f1_tr = f1_score(y_tr, clf.predict(X_tr), zero_division=0)
axes[0].plot(rec_tr, prec_tr, color='steelblue', linewidth=2)
axes[0].set_xlabel('Recall'); axes[0].set_ylabel('Precision')
axes[0].set_title(f'Train set - F1={f1_tr:.2f}  (n={len(y_tr)})')
print(f'Train F1={f1_tr:.4f}')

prec_te, rec_te, _ = precision_recall_curve(y_te, clf.predict_proba(X_te)[:,1])
f1_te = f1_score(y_te, clf.predict(X_te), zero_division=0)
axes[1].plot(rec_te, prec_te, color='coral', linewidth=2)
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title(f'Test set - F1={f1_te:.2f}  (n={len(y_te)})')
print(f'Test F1={f1_te:.4f}')

plt.tight_layout(); plt.show()

used 10 wikipedia pages, 5 about finance (relevant) and 5 about sports (irrelevant) for the financial markets query.

tfidf does most of the work since finance pages have a lot of the query words like stock, market and investment while the sports pages basically have none of them. pagerank scores are fairly similar across both groups. hits actually scores sports pages higher since they link to each other more within our graph, but the classifier still gets it right using tfidf.

the classifier got f1=1.0 on both train and test. this makes sense because the two groups are really easy to separate with these features. the pr curve looks step shaped because there are only 10 pages so each step is just one page crossing the threshold.